# 02 — V-JEPA-2 zero-shot na Read My Ears dataset

**Cel:** użyć foundation video model V-JEPA-2 (Meta, czerwiec 2025) **bez własnego treningu** jako ekstraktora cech, potem prosty klasyfikator na embeddingach. Sprawdzić czy SOTA video understanding bije ich custom heurystykę z Etap A (acc=0.583).

**Model:** `facebook/vjepa2-vitl-fpc16-256-ssv2` — V-JEPA-2 ViT-Large fine-tuned na Something-Something v2 (action recognition benchmark, 16 frames per clip).

**3 ścieżki ewaluacji:**
1. **k-NN** (k=5, 10) na embeddingach — czy klipy action vs background są w ogóle separowalne
2. **Linear probe** (logistic regression) — czy linearny klasyfikator wystarczy
3. **Comparison** vs Etap A movement-detection (acc=0.583) i paper claim (0.875)

**Brak GPU treningu** — tylko forward pass + sklearn na features.

## 1. Setup

In [ ]:
import os, sys, time
from pathlib import Path
import torch, cv2, numpy as np, pandas as pd

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)

print('Torch:', torch.__version__, '| MPS:', torch.backends.mps.is_available())
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', DEVICE)

DATA = POC / 'vendor/ReadMyEars_Dataset/data'
for split in ['train.csv', 'val.csv', 'test.csv']:
    df = pd.read_csv(DATA / split)
    avail = sum((DATA / v).exists() for v in df['video'])
    print(f'  {split}: {len(df)} klipów, {avail} lokalnie')

## 2. Załaduj V-JEPA-2 ViT-Large + processor

Model fine-tuned na SSv2 (action recognition) — najlepsze cechy motion. Pierwsze ładowanie pobiera ~1.2 GB z HF.

In [ ]:
from transformers import VJEPA2Model, AutoVideoProcessor

MODEL_ID = 'facebook/vjepa2-vitl-fpc16-256-ssv2'
t0 = time.time()
processor = AutoVideoProcessor.from_pretrained(MODEL_ID)
model = VJEPA2Model.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f'Załadowane w {time.time()-t0:.1f}s')
n_params = sum(p.numel() for p in model.parameters())
print(f'Parametry: {n_params/1e6:.0f}M')
print(f'Embedding dim:', model.config.hidden_size)
print(f'Frames per clip:', getattr(model.config, 'frames_per_clip', 16))

## 3. Helper: clip → 16 sample frames → V-JEPA-2 embedding

In [ ]:
def read_clip_frames(clip_path, num_frames=16):
    """Wczytaj N równomiernych klatek z mp4 jako tensor (T, H, W, C) RGB uint8."""
    cap = cv2.VideoCapture(str(clip_path))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total < 1:
        cap.release()
        return None
    indices = np.linspace(0, n_total - 1, num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, f = cap.read()
        if not ok:
            f = frames[-1] if frames else np.zeros((256, 256, 3), dtype=np.uint8)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames)  # (T, H, W, C)

@torch.no_grad()
def extract_embedding(clip_path, model, processor, num_frames=16):
    frames = read_clip_frames(clip_path, num_frames=num_frames)
    if frames is None:
        return None
    inputs = processor(videos=list(frames), return_tensors='pt').to(DEVICE)
    outputs = model(**inputs)
    # mean pool po patch tokens → vector
    last_hidden = outputs.last_hidden_state  # (B, N_tokens, D)
    emb = last_hidden.mean(dim=1).squeeze(0)
    return emb.cpu().float().numpy()

# Smoke test na jednym klipie
sample_clip = DATA / 'videos/action_S1.mp4_0_.mp4'
t0 = time.time()
emb = extract_embedding(sample_clip, model, processor)
print(f'Embedding shape: {emb.shape}, dtype: {emb.dtype}, czas: {time.time()-t0:.2f}s')

## 4. Ekstrakcja embeddingów dla wszystkich klipów (train + val + test)

In [ ]:
all_embs, all_labels, all_filenames, all_splits = [], [], [], []
t0 = time.time()
for split in ['train', 'val', 'test']:
    df = pd.read_csv(DATA / f'{split}.csv')
    n_done = 0
    for _, row in df.iterrows():
        clip = DATA / row['video']
        if not clip.exists():
            continue
        try:
            emb = extract_embedding(clip, model, processor)
            if emb is None:
                continue
            all_embs.append(emb)
            all_labels.append(1 if row['label'] == 'action' else 0)
            all_filenames.append(row['video'].split('/')[-1])
            all_splits.append(split)
            n_done += 1
            if n_done % 25 == 0:
                elapsed = time.time() - t0
                rate = (sum(s == split for s in all_splits)) / elapsed if elapsed > 0 else 0
                print(f'  {split} {n_done}/{len(df)} ({elapsed:.0f}s, {rate:.1f} klip/s)')
        except Exception as e:
            print(f'  ERR {row["video"]}: {e}')
    print(f'  ✓ {split} zakończone: {n_done} klipów')

print(f'\nTotal: {len(all_embs)} embeddingów w {time.time()-t0:.0f}s')
embs = np.stack(all_embs)
labels = np.array(all_labels)
splits = np.array(all_splits)
filenames = np.array(all_filenames)
np.savez(POC / 'outputs/vjepa2_embeddings.npz',
         embs=embs, labels=labels, splits=splits, filenames=filenames)
print(f'Zapisano: outputs/vjepa2_embeddings.npz ({embs.shape}, {embs.nbytes/1e6:.1f} MB)')

## 5. k-NN baseline

Sprawdzenie czy embeddingi action vs background są **w ogóle separowalne** w przestrzeni. Niski wynik = foundation model nie różnicuje tego task'u.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train na train+val, test na test
tr_mask = (splits == 'train') | (splits == 'val')
te_mask = splits == 'test'
X_tr, y_tr = embs[tr_mask], labels[tr_mask]
X_te, y_te = embs[te_mask], labels[te_mask]
print(f'Train+val: {len(X_tr)} klipów ({y_tr.sum()} action, {(y_tr==0).sum()} background)')
print(f'Test:      {len(X_te)} klipów ({y_te.sum()} action, {(y_te==0).sum()} background)')

# Normalizacja
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

knn_results = {}
for k in [1, 3, 5, 10, 15]:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn.fit(X_tr_s, y_tr)
    pred = knn.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred)
    knn_results[k] = {'acc': acc, 'f1': f1, 'pred': pred}
    print(f'  k={k:>2}: acc={acc:.3f}  f1={f1:.3f}')

best_k = max(knn_results, key=lambda k: knn_results[k]['acc'])
print(f'\nBest k={best_k}: accuracy={knn_results[best_k]["acc"]:.3f}')

## 6. Linear probe (logistic regression)

Najprostszy klasyfikator linearny. Trening na CPU w sekundach. Wynik = górna granica „co linearne separatory potrafią wycisnąć z embeddingów".

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_results = {}
for C in [0.01, 0.1, 1.0, 10.0]:
    lr = LogisticRegression(C=C, max_iter=2000, class_weight='balanced')
    lr.fit(X_tr_s, y_tr)
    pred = lr.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    p = precision_score(y_te, pred, zero_division=0)
    r = recall_score(y_te, pred)
    f1 = f1_score(y_te, pred)
    cm = confusion_matrix(y_te, pred)
    lr_results[C] = {'acc': acc, 'precision': p, 'recall': r, 'f1': f1, 'cm': cm, 'pred': pred}
    print(f'  C={C:>5.2f}: acc={acc:.3f}  prec={p:.3f}  rec={r:.3f}  f1={f1:.3f}')

best_C = max(lr_results, key=lambda c: lr_results[c]['acc'])
print(f'\nBest C={best_C}: accuracy={lr_results[best_C]["acc"]:.3f}')
print(f'Confusion matrix:\n{lr_results[best_C]["cm"]}')

## 7. Porównanie z Etap A i paper

In [ ]:
import matplotlib.pyplot as plt, json

# Etap A wyniki z poprzedniego notebooka
etap_a = json.load(open(POC / 'outputs/movement_detection_results.json'))

comparison = {
    'Paper claim (Alves 2025)': 0.875,
    'Etap A movement-detection': etap_a['accuracy'],
    f'V-JEPA-2 k-NN (k={best_k})': knn_results[best_k]['acc'],
    f'V-JEPA-2 Linear probe (C={best_C})': lr_results[best_C]['acc'],
}
print('=== Comparison Read My Ears ear-movement classification ===')
for k, v in comparison.items():
    print(f'  {k:40s} {v:.3f}')

fig, ax = plt.subplots(figsize=(9, 4))
names = list(comparison.keys())
vals = list(comparison.values())
colors = ['#888', '#d62728', '#1f77b4', '#2ca02c']
bars = ax.barh(names, vals, color=colors)
ax.axvline(0.5, color='red', linestyle=':', alpha=0.5, label='random (50%)')
ax.set_xlabel('Accuracy on test split (n=48)')
ax.set_xlim(0, 1)
ax.set_title('Read My Ears ear-movement classification — comparison')
for bar, val in zip(bars, vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(POC / 'outputs/vjepa2_comparison.png', dpi=120)
plt.show()

# Save final results
vjepa_results = {
    'model': MODEL_ID,
    'embedding_dim': int(embs.shape[1]),
    'n_train': int(len(X_tr)),
    'n_test': int(len(X_te)),
    'knn': {f'k_{k}': {'acc': float(v['acc']), 'f1': float(v['f1'])} for k, v in knn_results.items()},
    'best_knn': {'k': int(best_k), 'acc': float(knn_results[best_k]['acc']), 'f1': float(knn_results[best_k]['f1'])},
    'linear_probe': {f'C_{C}': {'acc': float(v['acc']), 'precision': float(v['precision']), 'recall': float(v['recall']), 'f1': float(v['f1'])} for C, v in lr_results.items()},
    'best_linear_probe': {'C': float(best_C), 'acc': float(lr_results[best_C]['acc']), 'precision': float(lr_results[best_C]['precision']), 'recall': float(lr_results[best_C]['recall']), 'f1': float(lr_results[best_C]['f1']), 'cm': lr_results[best_C]['cm'].tolist()},
    'comparison': {k: float(v) for k, v in comparison.items()},
}
with open(POC / 'outputs/vjepa2_results.json', 'w') as f:
    json.dump(vjepa_results, f, indent=2)
print(f'\nZapisano: outputs/vjepa2_results.json')

## 8. Notatka decyzyjna

**Co to mówi:**
- **acc ≥ 0.85** → V-JEPA-2 zerolot bije ich custom pipeline. Foundation models są realnym pivotem dla Faza 2/3 (24 RHpE behaviors zamiast trenowania per-class).
- **acc ≈ Etap A (0.58)** → problem jest w samym dataset (label noise lub task ill-defined). Replikacja paper'u jest specyficzna dla ich infrastruktury.
- **acc 0.65–0.80** → SOTA video models pomagają częściowo; ensemble z DLC keypoints + linear probe może być nasza droga.

**Co dalej (jeśli pozytywne):** sprawdzić X-CLIP text-conditioned zero-shot na 24 RHpE behaviors jako proof-of-concept dla multi-label scoring.